# Targeted training on the FinBERT transformer

Same question as before, now with the fine-tuned FinBERT transformer from notebook 05 as the model:
does giving hard training sentences more weight beat giving the same weight to random sentences?

1. **Hard sets:** 5-fold out-of-fold probabilities on train with the transformer, and the baseline's probabilities on val.
2. **Strategy 1, margin percentile:** hard = lowest p% of (top probability - second probability). A copy of the baseline is trained one more epoch with the hard sentences at weight 1.5.
3. **Strategy 2, confidence band + clean labels:** hard = top probability between 0.4 and 0.6, and only all-agree sentences get the weight K. Retrained from scratch.

Runtime on a Mac GPU (MPS) is about 2 hours with the default settings, about 6 minutes with `QUICK = True`.

## Settings

In [1]:
QUICK = False    # True: only check that the code runs. False: the real experiment

SEEDS = [42, 43, 44]
PERCENTILES = [10, 20, 30]   # strategy 1
KS = [5]                     # strategy 2
EPOCHS = 2                   # same as notebook 05
N_ROWS = None                # use all of train

if QUICK:
    SEEDS = [42]
    PERCENTILES = [20]
    EPOCHS = 1
    N_ROWS = 300

RESULTS_DIR = "../results/targeted_transformer"
if QUICK:
    RESULTS_DIR = RESULTS_DIR + "_quick"   # so test runs don't mix with the real results

## Imports

In [2]:
import copy
import os
import random

os.environ["HF_HOME"] = os.path.abspath("../.cache/huggingface")
os.environ["HF_HUB_CACHE"] = os.path.abspath("../.cache/huggingface/hub")

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from torch.optim import AdamW
from transformers import BertForSequenceClassification, BertTokenizerFast, get_linear_schedule_with_warmup

LABELS = ["negative", "neutral", "positive"]
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"   # mps = Mac GPU
print("device:", DEVICE)

os.makedirs(RESULTS_DIR + "/probs", exist_ok=True)

/Users/cleojj002/Documents/GitHub/nlp-financial-phrasebank/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1107 16:56:14.931000 63018 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


device: mps


## Data

`all_agree` comes from `results/hard_samples/agreement_*.csv` (notebook 07), which also matches the sentences with broken characters.

In [3]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
agree_train = pd.read_csv("../results/hard_samples/agreement_train.csv").all_agree.values
agree_val = pd.read_csv("../results/hard_samples/agreement_val.csv").all_agree.values

if N_ROWS is not None:
    train_df = train_df.iloc[:N_ROWS]
    agree_train = agree_train[:N_ROWS]

y_train = train_df.label.map({"negative": 0, "neutral": 1, "positive": 2}).values
y_val = val_df.label.map({"negative": 0, "neutral": 1, "positive": 2}).values

tokenizer = BertTokenizerFast.from_pretrained("yiyanghkust/finbert-pretrain")

def tokenize(df):
    tokens = tokenizer(list(df.sentence), padding="max_length", truncation=True,
                       max_length=64, return_tensors="pt")
    return tokens["input_ids"], tokens["attention_mask"]

ids_train, mask_train = tokenize(train_df)
ids_val, mask_val = tokenize(val_df)
print("train", ids_train.shape, "| val", ids_val.shape, "| all-agree train", agree_train.sum(), "val", agree_val.sum())

train torch.Size([2901, 64]) | val torch.Size([967, 64]) | all-agree train 1355 val 457


## Model

Same training as notebook 05, but each sentence can have a weight: loss = mean(weight * cross-entropy), like `sample_weight` in Keras.
We keep the epoch with the best val macro F1.

In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def new_model(seed):
    set_seed(seed)
    model = BertForSequenceClassification.from_pretrained("yiyanghkust/finbert-pretrain", num_labels=3)
    return model.to(DEVICE)


def predict(model, ids, mask):
    model.eval()
    probs = []
    with torch.no_grad():
        for start in range(0, len(ids), 64):
            logits = model(input_ids=ids[start:start + 64].to(DEVICE),
                           attention_mask=mask[start:start + 64].to(DEVICE)).logits
            probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(probs)


def train(model, ids, mask, labels, weights=None, epochs=EPOCHS, lr=2e-5, seed=42):
    if weights is None:
        weights = np.ones(len(labels))
    labels = torch.tensor(labels, dtype=torch.long)
    weights = torch.tensor(weights, dtype=torch.float32)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = (len(labels) // 16 + 1) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, steps // 10, steps)

    rng = np.random.default_rng(seed)
    best_f1 = -1
    for epoch in range(epochs):
        model.train()
        order = rng.permutation(len(labels))
        for start in range(0, len(order), 16):
            batch = order[start:start + 16]
            logits = model(input_ids=ids[batch].to(DEVICE), attention_mask=mask[batch].to(DEVICE)).logits
            losses = torch.nn.functional.cross_entropy(logits, labels[batch].to(DEVICE), reduction="none")
            loss = (losses * weights[batch].to(DEVICE)).mean()
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        val_f1 = f1_score(y_val, predict(model, ids_val, mask_val).argmax(axis=1), average="macro")
        print(f"   epoch {epoch + 1}/{epochs}: val macro F1 {val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model


def score(probs, hard_val):
    # accuracy and macro F1 on all val sentences and on the subsets, plus F1 per class
    pred = probs.argmax(axis=1)
    subsets = {"overall": np.ones(len(y_val), dtype=bool), "hard": hard_val, "nonhard": ~hard_val,
               "allagree": agree_val, "not_allagree": ~agree_val}
    scores = {}
    for name in subsets:
        mask = subsets[name]
        scores["accuracy_" + name] = accuracy_score(y_val[mask], pred[mask])
        scores["f1_" + name] = f1_score(y_val[mask], pred[mask], average="macro", zero_division=0)
    per_class = f1_score(y_val, pred, labels=[0, 1, 2], average=None, zero_division=0)
    scores["f1_negative"] = per_class[0]
    scores["f1_neutral"] = per_class[1]
    scores["f1_positive"] = per_class[2]
    return scores


def save_rows(rows, name):
    # one csv per strategy; running a setting again replaces its old rows
    path = f"{RESULTS_DIR}/{name}.csv"
    new = pd.DataFrame(rows)
    if os.path.exists(path):
        new = pd.concat([pd.read_csv(path), new], ignore_index=True)
    new = new.drop_duplicates(subset=["arm", "param", "seed"], keep="last")
    new.to_csv(path, index=False)
    return new

## 1. Hard sets

Every train sentence is predicted by a model that did not see it (5 folds). Val is predicted by the baseline trained on all of train.
Saved to `results/hard_samples/transformer_band_train.csv` / `_val.csv` (same columns as the CNN band files). The folds take ~15 min, so if the file is already there we just load it.

In [5]:
prob_cols = ["p_negative", "p_neutral", "p_positive"]
train_path = "../results/hard_samples/transformer_band_train.csv"
val_path = "../results/hard_samples/transformer_band_val.csv"

if os.path.exists(train_path) and not QUICK:
    oof = pd.read_csv(train_path)[prob_cols].values
    print("loaded", train_path)
else:
    oof = np.zeros((len(y_train), 3))
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for fold_train, fold_test in folds.split(ids_train, y_train):
        print("fold")
        fold_model = train(new_model(42), ids_train[fold_train], mask_train[fold_train], y_train[fold_train])
        oof[fold_test] = predict(fold_model, ids_train[fold_test], mask_train[fold_test])
        del fold_model

fold


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 41500.92it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: yiyanghkust/finbert-pretrain
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING

   epoch 1/2: val macro F1 0.8135
   epoch 2/2: val macro F1 0.8200
fold


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 34705.47it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: yiyanghkust/finbert-pretrain
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING

   epoch 1/2: val macro F1 0.8001
   epoch 2/2: val macro F1 0.8221
fold


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 59828.43it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: yiyanghkust/finbert-pretrain
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING

   epoch 1/2: val macro F1 0.8070
   epoch 2/2: val macro F1 0.8229
fold


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 58912.09it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: yiyanghkust/finbert-pretrain
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING

   epoch 1/2: val macro F1 0.7787
   epoch 2/2: val macro F1 0.8061
fold


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 50250.84it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: yiyanghkust/finbert-pretrain
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING

   epoch 1/2: val macro F1 0.8078
   epoch 2/2: val macro F1 0.8171


In [6]:
print("baseline (all train)")
baseline = train(new_model(42), ids_train, mask_train, y_train)
val_probs = predict(baseline, ids_val, mask_val)
print("baseline val macro F1:", round(f1_score(y_val, val_probs.argmax(axis=1), average="macro"), 4))

baseline (all train)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 47100.42it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: yiyanghkust/finbert-pretrain
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING

KeyboardInterrupt: 

In [ ]:
def band_table(df, probs, agree):
    out = df[["sentence", "label"]].copy()
    out["p_negative"] = probs[:, 0]
    out["p_neutral"] = probs[:, 1]
    out["p_positive"] = probs[:, 2]
    out["pred"] = [LABELS[i] for i in probs.argmax(axis=1)]
    out["max_prob"] = probs.max(axis=1)
    out["is_hard"] = (out.max_prob >= 0.4) & (out.max_prob <= 0.6)
    out["all_agree"] = agree
    return out

band_train = band_table(train_df, oof, agree_train)
band_val = band_table(val_df, val_probs, agree_val)
if not QUICK:
    band_train.to_csv(train_path, index=False)
    band_val.to_csv(val_path, index=False)

print("OOF macro F1 on train:", round(f1_score(y_train, oof.argmax(axis=1), average="macro"), 4))
print("in the band: train", band_train.is_hard.sum(), "| val", band_val.is_hard.sum())

## 2. Strategy 1: margin percentile

Hard = lowest p% of the train margins; the same cutoff is used on val.
For every seed a copy of the baseline is trained one more epoch (lr 1e-5) with the hard sentences (targeted) or the same number of random sentences (random) at weight 1.5.

In [ ]:
def margins(probs):
    sorted_probs = np.sort(probs, axis=1)
    return sorted_probs[:, -1] - sorted_probs[:, -2]

margin_train = margins(oof)
margin_val = margins(val_probs)


def fine_tune(weighted_idx, seed):
    set_seed(seed)
    model = copy.deepcopy(baseline)   # a copy, so the baseline itself does not change
    weights = np.ones(len(y_train))
    weights[weighted_idx] = 1.5
    return train(model, ids_train, mask_train, y_train, weights, epochs=1, lr=1e-5, seed=seed)


rows = []
for p in PERCENTILES:
    cutoff = np.percentile(margin_train, p)
    hard_idx = np.where(margin_train <= cutoff)[0]
    hard_val = margin_val <= cutoff
    print(f"P{p}: {len(hard_idx)} hard train, {hard_val.sum()} hard val (cutoff {cutoff:.3f})")

    for seed in SEEDS:
        random_idx = np.random.default_rng(seed).choice(len(y_train), len(hard_idx), replace=False)
        for arm, idx in [("targeted", hard_idx), ("random", random_idx)]:
            model = fine_tune(idx, seed)
            probs = predict(model, ids_val, mask_val)
            del model
            np.save(f"{RESULTS_DIR}/probs/s1_{arm}_p{p}_s{seed}.npy", probs)
            row = {"arm": arm, "param": p, "seed": seed, "n_weighted": len(idx), "n_hard_val": int(hard_val.sum())}
            row.update(score(probs, hard_val))
            rows.append(row)
            print(f"   {arm} seed={seed}: f1={row['f1_overall']:.3f} hard={row['f1_hard']:.3f}")

strategy1 = save_rows(rows, "strategy1")

## 3. Strategy 2: confidence band + clean labels

Hard = top probability between 0.4 and 0.6. Only the hard sentences where all annotators agreed are up-weighted (clean-hard). Every run is trained from scratch.

| arm | sentences with weight K |
|---|---|
| no_weight | none (reference) |
| clean_hard | hard and all-agree |
| random | same number, random from all train |
| clean_random | same number, random from all-agree sentences that are not hard |
| variant | clean_hard gets K, hard sentences without full agreement get 0.5 |

The transformer is more sure of itself than the CNN, so expect fewer sentences in the band. Check the counts below.

In [ ]:
hard_train = band_train.is_hard.values
hard_val_s2 = band_val.is_hard.values

clean_hard = np.where(hard_train & agree_train)[0]
ambiguous_hard = np.where(hard_train & ~agree_train)[0]
clean_easy = np.where(~hard_train & agree_train)[0]
print("hard train:", hard_train.sum(), "| clean-hard:", len(clean_hard), "| ambiguous hard:", len(ambiguous_hard),
      "| hard val:", hard_val_s2.sum())


def strategy2_weights(arm, k, seed):
    rng = np.random.default_rng(seed)
    weights = np.ones(len(y_train))
    if arm == "clean_hard":
        weights[clean_hard] = k
    elif arm == "random":
        weights[rng.choice(len(y_train), len(clean_hard), replace=False)] = k
    elif arm == "clean_random":
        n = min(len(clean_hard), len(clean_easy))   # only matters in QUICK mode
        weights[rng.choice(clean_easy, n, replace=False)] = k
    elif arm == "variant":
        weights[ambiguous_hard] = 0.5
        weights[clean_hard] = k
    return weights


def run_strategy2(arm, k, seed):
    print(f"{arm} k={k} seed={seed}")
    model = train(new_model(seed), ids_train, mask_train, y_train, strategy2_weights(arm, k, seed), seed=seed)
    probs = predict(model, ids_val, mask_val)
    del model
    np.save(f"{RESULTS_DIR}/probs/s2_{arm}_k{k}_s{seed}.npy", probs)
    row = {"arm": arm, "param": k, "seed": seed, "n_weighted": len(clean_hard), "n_hard_val": int(hard_val_s2.sum())}
    row.update(score(probs, hard_val_s2))
    print(f"   f1={row['f1_overall']:.3f} hard={row['f1_hard']:.3f}")
    return row

In [ ]:
rows = []
if len(clean_hard) == 0:
    print("no clean-hard sentences, strategy 2 skipped")
else:
    for seed in SEEDS:
        rows.append(run_strategy2("no_weight", 1, seed))
    for k in KS:
        for arm in ["clean_hard", "random", "clean_random", "variant"]:
            for seed in SEEDS:
                rows.append(run_strategy2(arm, k, seed))
    strategy2 = save_rows(rows, "strategy2")

## Compare

Mean and std over the seeds, and targeted minus its control seed by seed (both use the same seeds). A difference smaller than the std is not an effect.

In [ ]:
cols = ["f1_overall", "f1_hard", "f1_nonhard", "f1_allagree", "f1_not_allagree"]

s1 = pd.read_csv(f"{RESULTS_DIR}/strategy1.csv")
s1.groupby(["param", "arm"])[cols].agg(["mean", "std"]).round(3)

In [ ]:
s2 = pd.read_csv(f"{RESULTS_DIR}/strategy2.csv")
s2.groupby(["arm", "param"])[cols].agg(["mean", "std"]).round(3)

In [ ]:
def paired_diff(df, target, control, param):
    a = df[(df.arm == target) & (df.param == param)].sort_values("seed").f1_overall.values
    b = df[(df.arm == control) & (df.param == param)].sort_values("seed").f1_overall.values
    if len(a) == 0 or len(a) != len(b):
        return
    diff = a - b
    std = diff.std(ddof=1) if len(diff) > 1 else float("nan")
    print(f"{target} - {control}, param={param}: mean {diff.mean():+.3f}, std {std:.3f}, per seed {np.round(diff, 3).tolist()}")

print("Strategy 1")
for p in sorted(s1.param.unique()):
    paired_diff(s1, "targeted", "random", p)

print("\nStrategy 2")
for k in sorted(s2[s2.arm != "no_weight"].param.unique()):
    for control in ["random", "clean_random"]:
        paired_diff(s2, "clean_hard", control, k)